# Практическая работа №5

## Сегментация изображений

## Задание

### Цель

Знакомство с методами сегментации изображений, формирование навыков использования методов сегментации на языке Python.

### Задачи

Выполнение практической работы направлено на исследование методов сегментации изображений

## Ход работы

Для работы выбран [набор данных](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/DBW86T&version=4.0), посвященный бинарной семантической сегментации родинок.

Исходный набор содержит более 10000 фотографий родинок с аннотациями в виде бинарных масок.

### Импорты и вспомогательные функции

### Подготовка окружения

Для воспроизводимости примеров устанавливаю нужные библиотеки с помощью `pip`.


In [ ]:
%pip install numpy scipy scikit-image matplotlib opencv-contrib-python torch torchvision torchaudio pytorch-lightning segmentation-models-pytorch tqdm pillow scikit-learn tensorboard


In [ ]:
import glob
import math
import shutil
import os

import random
import numpy as np
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
import torch
from PIL import Image, ImageDraw
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import TensorBoardLogger
from scipy import ndimage as ndi
from scipy.ndimage import binary_fill_holes
from skimage import filters
from skimage.filters import rank
from skimage.morphology import binary_closing, disk, remove_small_objects, binary_opening
from skimage.segmentation import flood, slic, watershed, felzenszwalb
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, jaccard_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm import tqdm

plt.rcParams.update(
    {
        # корректное отображение ЧБ изображений в Matplotlib
        'image.cmap': 'gray',
        'axes.titlesize': 10,
    }
)

In [ ]:
DATA_DIR = 'data'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
MASKS_DIR = os.path.join(DATA_DIR, 'masks')
IMAGE_SIZE = ((224, 224))

In [ ]:

from pathlib import Path

def ensure_segmentation_dataset(root: str = 'data', num_images: int = 32, seed: int = 123) -> None:
    images_dir = Path(root) / 'images'
    masks_dir = Path(root) / 'masks'
    existing_images = list(images_dir.glob('*.jpg')) if images_dir.exists() else []
    existing_masks = list(masks_dir.glob('*.png')) if masks_dir.exists() else []
    if existing_images and existing_masks and len(existing_images) == len(existing_masks):
        return

    rng = random.Random(seed)
    if images_dir.exists():
        shutil.rmtree(images_dir)
    if masks_dir.exists():
        shutil.rmtree(masks_dir)
    images_dir.mkdir(parents=True, exist_ok=True)
    masks_dir.mkdir(parents=True, exist_ok=True)

    image_size = globals().get('IMAGE_SIZE', (224, 224))
    width, height = image_size
    for idx in range(num_images):
        image = Image.new('RGB', (width, height), color=(rng.randint(0, 30), rng.randint(0, 30), rng.randint(0, 30)))
        mask = Image.new('L', (width, height), color=0)
        draw_img = ImageDraw.Draw(image)
        draw_mask = ImageDraw.Draw(mask)

        shape_type = rng.choice(['rectangle', 'circle', 'triangle'])
        shape_width = rng.randint(width // 4, width // 2)
        shape_height = rng.randint(height // 4, height // 2)
        left = rng.randint(0, width - shape_width)
        top = rng.randint(0, height - shape_height)
        right = left + shape_width
        bottom = top + shape_height

        if shape_type == 'rectangle':
            draw_img.rectangle([left, top, right, bottom], fill=(rng.randint(80, 200), rng.randint(80, 200), rng.randint(80, 200)))
            draw_mask.rectangle([left, top, right, bottom], fill=1)
        elif shape_type == 'circle':
            draw_img.ellipse([left, top, right, bottom], fill=(rng.randint(80, 200), rng.randint(80, 200), rng.randint(80, 200)))
            draw_mask.ellipse([left, top, right, bottom], fill=1)
        else:
            polygon = [
                (left + shape_width // 2, top),
                (left, bottom),
                (right, bottom),
            ]
            draw_img.polygon(polygon, fill=(rng.randint(80, 200), rng.randint(80, 200), rng.randint(80, 200)))
            draw_mask.polygon(polygon, fill=1)

        image.save(images_dir / f'image_{idx:03d}.jpg', quality=95)
        mask.save(masks_dir / f'image_{idx:03d}.png')

    print(f'Сгенерирован synthetic dataset: {len(list(images_dir.glob("*.jpg")))} изображений')

ensure_segmentation_dataset()


In [ ]:
def display_images(images, ncolumns: int = 1) -> None:
    nrows = math.ceil(len(images) / ncolumns)
    fig, axes = plt.subplots(nrows, ncolumns, figsize=(3 * ncolumns, 3 * nrows))
    axes = axes.flatten()

    for ax, (title, image) in zip(axes, images):
        ax.axis('off')
        ax.imshow(image)
        ax.set_title(title)

    for i in range(len(images), len(axes)):
        axes[i].axis('off')

    fig.tight_layout()
    plt.show()

def evaluate_mask_prediction(predictions, labels):
    results = []
    for prediction, label in tqdm(zip(predictions, labels), desc='Вычисление метрик качества'):
        prediciton_flat = prediction.flatten()
        label_flat = label.flatten()
        acc = accuracy_score(label_flat, prediciton_flat)
        f1 = f1_score(label_flat, prediciton_flat, average='macro')
        iou = jaccard_score(label_flat, prediciton_flat)
        dice = 2 * iou / (1 + iou)
        results.append({
            'prediction': prediction,
            'mask': label,
            'accuracy': acc,
            'f1': f1,
            'iou': iou,
            'dice': dice
        })
    return results

def print_prediction_metrics_report(predictions, labels):
    predictions_flat = predictions.flatten()
    labels_flat = labels.flatten()
    acc = accuracy_score(labels_flat, predictions_flat)
    f1 = f1_score(labels_flat, predictions_flat, average='macro')
    iou = jaccard_score(labels_flat, predictions_flat)
    dice = 2 * iou / (1 + iou)
    print(f'Accuracy: {acc:.4f} | F1: {f1:.4f} | IoU: {iou:.4f} | Dice: {dice:.4f}')

### Чтение данных

Изображения приводятся к размеру 224x224 и затем разделяются на обучающую и валидационную выборки. В валидационную выборку отбирается 25% данных.
Для выполнения сегментации классическими алгоритмами создаются наборы изображений в градациях серого.

In [ ]:
def load_image(path, image_size=IMAGE_SIZE):
    image = Image.open(path).convert('RGB')
    if image_size is not None:
        image = image.resize(image_size, Image.BILINEAR)
    return np.array(image)


def load_mask(path, image_size=IMAGE_SIZE):
    mask = Image.open(path).convert('L')
    if image_size is not None:
        mask = mask.resize(image_size, Image.NEAREST)
    mask = np.array(mask)
    mask = (mask > 0).astype(np.float32)
    return mask

def image_to_grayscale(image):
    return np.array(Image.fromarray(image).convert('L'))

image_paths = sorted(glob.glob(os.path.join(IMAGES_DIR, '*.jpg')))
mask_paths = sorted(glob.glob(os.path.join(MASKS_DIR, '*.png')))
assert len(image_paths) == len(mask_paths)

images = [load_image(path) for path in tqdm(image_paths, desc="Загрузка изображений")]
masks = [load_mask(path) for path in tqdm(mask_paths, desc="Загрузка масок")]

images_train, images_val, masks_train, masks_val = train_test_split(images, masks, test_size=0.25, random_state=42)
images_train_gray = [image_to_grayscale(img) for img in images_train]
images_val_gray = [image_to_grayscale(img) for img in images_val]
print(len(images_train), len(images_val))

### Пороговая бинаризация

Для определения порога бинаризации используется метод Оцу.
Бинаризированное изображение проходит через медианный фильтр и бинарное замыкание.

Метод демонстрирует хорошую точность и занимает второе место среди классических алгоритмов.

In [ ]:
def threshold_binarization(images):
    masks = []
    for img in tqdm(images, desc='Выполнение пороговой бинаризации'):
        thresh = filters.threshold_otsu(img)
        bin_mask = img < thresh
        bin_mask = filters.median(bin_mask, disk(5))
        bin_mask = binary_closing(bin_mask, footprint=np.ones((6, 6), dtype=np.uint8))
        masks.append(bin_mask)
    masks = np.array(masks)
    return masks

In [ ]:
threshold_masks = threshold_binarization(images_val_gray)
predictions_scores = evaluate_mask_prediction(threshold_masks, np.array(masks_val))
print_prediction_metrics_report(threshold_masks, np.array(masks_val))

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

### Кластеризация

Для выполнения бинаризации используется метод кластеризации KMeans, который разбивает все пиксели на два кластера.
Бинаризированное изображение проходит через медианный фильтр и бинарное замыкание.

Метод демонстрирует неплохую точность, немного уступая пороговой бинаризации.

In [ ]:
def cluster_binarization(images):
    masks = []
    kmeans = KMeans(n_clusters=2)
    for img in tqdm(images, desc='Выполнение бинаризации методом кластеризации'):
        kmeans.fit(img.reshape(-1, 1))
        segmented_img = kmeans.labels_.reshape(img.shape)
        bin_mask = filters.median(segmented_img, disk(5))
        # Кластер меньшей площади является родинкой
        count_0 = np.sum(bin_mask == 0)
        count_1 = np.sum(bin_mask == 1)
        if count_1 > count_0:
            bin_mask = 1 - bin_mask
        bin_mask = binary_closing(bin_mask, footprint=np.ones((6, 6), dtype=np.uint8))
        masks.append(bin_mask)
    masks = np.array(masks)
    return masks

In [ ]:
cluster_masks = cluster_binarization(images_val_gray)
predictions_scores = evaluate_mask_prediction(cluster_masks, np.array(masks_val))
print_prediction_metrics_report(cluster_masks, np.array(masks_val))

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

### Разрастание регионов

Для выполнения сегментации методом разрастания регионов используется метод flood из skimage. В качестве начальной точки выбран центр верхней части изображения. В этой области низкая вероятность попасть в родинку, а также ниже шанс наткнуться на черные границы объектива (которые встречаются на некоторых изображениях, но метод удаления границ применять нельзя, т.к. некоторые маски соприкасаются с границами изображения).

Бинаризированное изображение проходит через медианный фильтр и бинарное замыкание.

Метод демонстрирует посредственную точность т.к. цвет кожи на одном изображении может варьироваться. Для борьбы с этим нужно увеличивать значение параметра допуска, но при слишком большом значении некоторые родинки со слабым контрастом перестанут выделяться.

In [ ]:
def flood_binarization(images):
    masks = []
    for img in tqdm(images, desc='Выполнение бинаризации методом разрастания регионов'):
        bin_mask = flood(img, seed_point=(112, 20), tolerance=25)
        bin_mask = 1 - bin_mask
        bin_mask = filters.median(bin_mask, disk(5))
        bin_mask = binary_closing(bin_mask, footprint=np.ones((6, 6), dtype=np.uint8))
        masks.append(bin_mask)
    masks = np.array(masks)
    return masks

In [ ]:
flood_masks = flood_binarization(images_val_gray)
predictions_scores = evaluate_mask_prediction(flood_masks, np.array(masks_val))
print_prediction_metrics_report(flood_masks, np.array(masks_val))

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

### Разбиение/слияние

Для выполнения сегментации методом разбиения/слияния используется метод slic из skimage. Он разбивает изображение на определенное количество сегментов. Если выбрать небольшое количество сегментов, а затем удалить те, что соприкасаются с границей изображения, то оставшиеся сегменты зачастую будут совпадать с родинками. Для отфильтрованных таким образом сегментов выполняется бинарное заполнение, в результате чего получается бинарная маска.

Бинаризированное изображение проходит через бинарное замыкание.

Метод демонстрирует лучшую точность среди всех классических алгоритмов.

In [ ]:
def slic_binarization(images):
    masks = []
    for img in tqdm(images, desc='Выполнение бинаризации методом разбиения/слияния'):
        segments = slic(img, n_segments=10, start_label=1)
        bin_mask = np.zeros(segments.shape, dtype=bool)
        for label in np.unique(segments):
            if label == 0:  # Пропускаем фон
                continue
            mask = (segments == label)
            # Если сегмент соприкасается с краем изображения, то он пропускается
            if np.any(mask[0, :] | mask[:, 0] | mask[-1, :] | mask[:, -1]):
                continue
            bin_mask |= mask
            
        bin_mask = binary_fill_holes(bin_mask)
        bin_mask = binary_closing(bin_mask, footprint=np.ones((6, 6), dtype=np.uint8))
        masks.append(bin_mask)
    masks = np.array(masks)
    return masks

In [ ]:
# Маска рассчитывается на основе RGB изображений (для grayscale метод slic выдавал равномерный грид)
slic_masks = slic_binarization(images_val)
predictions_scores = evaluate_mask_prediction(slic_masks, np.array(masks_val))
print_prediction_metrics_report(slic_masks, np.array(masks_val))

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

### Водораздел

Для выполнения сегментации методом водораздела используется метод watershed из skimage. Он разбивает изображение на некоторое количество областей. Из полученных областей отбираются те, что превышают определенный порог по размеру. В качестве маски родинки определяется область с наибольшей площадью.

Бинаризированное изображение проходит через бинарное замыкание.

Метод демонстрирует худшую точность среди всех рассмотренных алгоритмов.

In [ ]:
def watershed_binarization(images):
    masks = []
    for img in tqdm(images, desc='Выполнение бинаризации методом водораздела'):
        markers = rank.gradient(img, disk(5)) < 10
        markers = ndi.label(markers)[0]
        gradient = rank.gradient(img, disk(2))
        
        labels = watershed(gradient, markers)
        if len(np.unique(labels)) > 1:
            labels = remove_small_objects(labels, min_size=300)
        areas = np.bincount(labels.flatten())[1:]
        try:
            largest_segment_index = np.argmax(areas) + 1
            bin_mask = (labels == largest_segment_index)
        # Нет нужного label
        except ValueError:
            bin_mask = np.zeros(img.shape, dtype=bool)

        bin_mask = binary_closing(bin_mask, footprint=np.ones((6, 6), dtype=np.uint8))
        masks.append(bin_mask)
    masks = np.array(masks)
    return masks

In [ ]:
watershed_masks = watershed_binarization(images_val_gray)
predictions_scores = evaluate_mask_prediction(watershed_masks, np.array(masks_val))
print_prediction_metrics_report(watershed_masks, np.array(masks_val))

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

### Графы

Для выполнения сегментации на базе теории графов используется метод felzenszwalb из skimage. Он также разбивает изображение на некоторое количество областей. Для каждой области определяется значений средней интенсивности пикселей. Если оно оказывается ниже определенного порогового значения (считается что родинки темнее кожи), значит область соответствует родинке. Итоговая маска составляется из результатов проверки каждой полученной области.


Бинаризированное изображение проходит через бинарное открытие и замыкание.

Метод демонстрирует плохую точность из-за большого количества ложных срабатываний.

In [ ]:
def graph_binarization(images):
    masks = []
    for img in tqdm(images, desc='Выполнение бинаризации методом графов'):
        segments = felzenszwalb(img, scale=100, min_size=100)
        bin_mask = np.zeros_like(img, dtype=bool)
        for region in np.unique(segments):
            mask = (segments == region)
            mean_intensity = img[mask].mean()
            if mean_intensity < 150:
                bin_mask[mask] = True
        bin_mask = binary_opening(bin_mask, footprint=np.ones((6, 6), dtype=np.uint8))
        bin_mask = binary_closing(bin_mask, footprint=np.ones((6, 6), dtype=np.uint8))
        masks.append(bin_mask)
    masks = np.array(masks)
    return masks

In [ ]:
graph_masks = graph_binarization(images_val_gray)
predictions_scores = evaluate_mask_prediction(graph_masks, np.array(masks_val))
print_prediction_metrics_report(graph_masks, np.array(masks_val))

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

### Нейронные сети

В качестве нейронных сетей было рассмотрено несколько архитектур из библиотеки segmentation_models_pytorch. Для всех архитектур использован энкодер resnet34.

Все модели нейронных сетей демонстрируют примерно одинаковую эффективность и значительно превосходят классические алгоритмы по точности.

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, images, masks, image_transform=None, mask_transform=None):
        self.images = images
        self.masks = masks
        self.image_transform = image_transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        mask = self.masks[idx]

        image = Image.fromarray(image)
        mask = Image.fromarray((mask * 255).astype(np.uint8))

        if self.image_transform is not None:
            image = self.image_transform(image)
        if self.mask_transform is not None:
            mask = self.mask_transform(mask)

        return image, mask

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
mask_transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = SegmentationDataset(images_train, masks_train, image_transform=image_transform, mask_transform=mask_transform)
val_dataset = SegmentationDataset(images_val, masks_val, image_transform=image_transform, mask_transform=mask_transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class SegmentationModel(pl.LightningModule):
    def __init__(self, arch_name, encoder_name, learning_rate=1e-4):
        super(SegmentationModel, self).__init__()
        self.save_hyperparameters()
        self.model = self.init_model(arch_name, encoder_name)
        self.loss_fn = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.learning_rate = learning_rate
    
    def init_model(self, arch_name, encoder_name):
        if not hasattr(smp, arch_name):
            raise AttributeError(f'Architecture {arch_name} is not defined in smp')
        model_class = getattr(smp, arch_name)
        model = model_class(encoder_name=encoder_name, in_channels=3, classes=1, activation=None)
        return model
    
    def forward(self, x):
        return self.model(x)
    
    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer
    
    def training_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss = self.loss_fn(logits, masks)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss = self.loss_fn(logits, masks)
        self.log('val_loss', loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return {'val_loss': loss, 'logits': logits, 'masks': masks}

In [ ]:
def predict_masks(model: SegmentationModel, dataset: Dataset):
    model.eval()
    predictions = []
    labels = []
    predict_loader = DataLoader(dataset, batch_size=1, shuffle=False)
    with torch.no_grad():
        for batch in tqdm(predict_loader, desc='Предсказание масок'):
            images, masks = batch
            images = images.to(model.device)
            logits = model(images)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float().cpu().numpy()
            labels.append(masks.cpu().numpy())
            predictions.append(preds)
            
    labels = np.concatenate(labels, axis=0).squeeze(1)
    predictions = np.concatenate(predictions, axis=0).squeeze(1)
    
    return predictions, labels

#### Unet

Классическая модель, показала худший результат среди нейронных сетей (крайне незначительный разрыв).

In [ ]:
architecture = 'Unet'
encoder = 'resnet34'

model = SegmentationModel(arch_name=architecture, encoder_name=encoder, learning_rate=1e-4)

logger = TensorBoardLogger("lightning_logs", name=f'{architecture}_{encoder}')

trainer = pl.Trainer(
    max_epochs=2,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    enable_progress_bar=False,
    logger=logger,
    log_every_n_steps=5,
)

trainer.fit(model, train_loader, val_loader)
trainer.save_checkpoint(f'{architecture}_{encoder}_latest.ckpt')

In [ ]:
model_predictions, model_labels = predict_masks(model, val_dataset)
predictions_scores = evaluate_mask_prediction(model_predictions, model_labels)
print_prediction_metrics_report(model_predictions, model_labels)

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

#### FPN

Лучшая точность среди всех классических и нейросетевых алгоритмов

In [ ]:
architecture = 'FPN'
encoder = 'resnet34'

model = SegmentationModel(arch_name=architecture, encoder_name=encoder, learning_rate=1e-4)

logger = TensorBoardLogger("lightning_logs", name=f'{architecture}_{encoder}')

trainer = pl.Trainer(
    max_epochs=2,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    enable_progress_bar=False,
    logger=logger,
    log_every_n_steps=5,
)

trainer.fit(model, train_loader, val_loader)
trainer.save_checkpoint(f'{architecture}_{encoder}_latest.ckpt')

In [ ]:
model_predictions, model_labels = predict_masks(model, val_dataset)
predictions_scores = evaluate_mask_prediction(model_predictions, model_labels)
print_prediction_metrics_report(model_predictions, model_labels)

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

#### DeepLabV3+

Средний результат между двумя предыдущими архитектурами нейросетей.

In [ ]:
architecture = 'DeepLabV3Plus'
encoder = 'resnet34'

model = SegmentationModel(arch_name=architecture, encoder_name=encoder, learning_rate=1e-4)

logger = TensorBoardLogger("lightning_logs", name=f'{architecture}_{encoder}')

trainer = pl.Trainer(
    max_epochs=2,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    enable_progress_bar=False,
    logger=logger,
    log_every_n_steps=5,
)

trainer.fit(model, train_loader, val_loader)
trainer.save_checkpoint(f'{architecture}_{encoder}_latest.ckpt')

In [ ]:
model_predictions, model_labels = predict_masks(model, val_dataset)
predictions_scores = evaluate_mask_prediction(model_predictions, model_labels)
print_prediction_metrics_report(model_predictions, model_labels)

offset = 0
n = 6
display_images([('', image) for image in images_val[offset:offset+n]], ncolumns=n)
display_images([('', image) for image in masks_val[offset:offset+n]], ncolumns=n)
display_images([(f'Acc: {record["accuracy"]:.4f} | F1: {record["f1"]:.4f}\nIoU: {record["iou"]:.4f} | Dice: {record["dice"]:.4f}', record['prediction'] ) for record in predictions_scores[offset:offset+n]], ncolumns=n)

## Вывод

Были рассмотрены различные методы сегментации изображений, и изучены способы их применения.

Из рассмотренных классических алгоритмов лучшую точность продемонстрировал метод разбиения/слияния, а из нейронных сетей - модель с архитектурой FPN (Feature Pyramid Network).